In [1]:
!pip install mcp

In [2]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("UtilityServer")

server_code = """
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("UtilityServer")

@mcp.tool()
def calculate(expression: str) -> str:
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@mcp.tool()
def get_weather(city: str) -> str:
    weather_data = {
        "bangalore": "28°C, Cloudy",
        "mumbai": "31°C, Humid",
        "delhi": "36°C, Sunny"
    }
    return weather_data.get(city.lower().strip(), "weather information not available")

if __name__ == "__main__":
    mcp.run()
"""

with open("mcp_server.py", "w") as f:
    f.write(server_code.strip())

In [4]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import asyncio
import os

async def agent(user_message):
    mcp_server_params = StdioServerParameters(
        command="python",
        args=["mcp_server.py"]
    )

    with open(os.devnull, 'w') as devnull_file:
        async with stdio_client(mcp_server_params, errlog=devnull_file) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()

                if any(on in user_message for on in ["+", "-", "*", "/"]):
                    result = await session.call_tool(
                        "calculate",
                        {"expression": user_message}
                    )
                    print("Result:", result.content[0].text)

                elif "weather" in user_message.lower():
                    city = user_message.lower().replace("weather", "").strip()
                    result = await session.call_tool(
                        "get_weather",
                        {"city": city}
                    )
                    print("Weather:", result.content[0].text)

                else:
                    print("Unable to determine tool")

query = input("Enter query: ")
await agent(query)

Enter query: 3+4
Result: 7
